# Truck Blind Spot Detection - App Runner

Notebook này gọi thẳng `app.main()` bằng GPU. Toàn bộ logic demo vẫn nằm trong `app.py`; notebook không chạy pipeline riêng.

| Thành phần | Giá trị |
|---|---|
| Script | `app.py` |
| Device Colab | `--device 0` khi có CUDA |
| Weights | `weights/best_roiv2.pt` |
| Video demo | `assets/videos/demo4.mp4` |

Vì Colab không có cửa sổ X11, `cv2.imshow` và `cv2.waitKey` được thay bằng no-op để pipeline chạy đầy đủ trên GPU mà không cần màn hình.


## 0. Chuẩn bị môi trường

Cell này tự nhận diện repo hiện tại. Nếu đang ở Colab và chưa có repo, cell sẽ clone project vào `/content/truck_blind_spot` và cài dependencies Python.


In [ ]:
from __future__ import annotations

import os
import shlex
import subprocess
import sys
from pathlib import Path
from typing import Optional

REPO_URL = "https://github.com/VTD0102/truck_blind_spot.git"
COLAB_PROJECT_ROOT = Path("/content/truck_blind_spot")

try:
    import google.colab  # type: ignore[import-not-found]

    IS_COLAB = True
except ImportError:
    IS_COLAB = False


def run_command(command: list[str], cwd: Optional[Path] = None) -> None:
    """Chạy lệnh shell và dừng notebook nếu lệnh thất bại."""
    print("$ " + " ".join(shlex.quote(part) for part in command))
    subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        check=True,
    )


current_dir = Path.cwd().resolve()
if not (current_dir / "app.py").exists() or not (current_dir / "src" / "pipeline.py").exists():
    if IS_COLAB:
        if not COLAB_PROJECT_ROOT.exists():
            run_command(["git", "clone", REPO_URL, str(COLAB_PROJECT_ROOT)])
        os.chdir(COLAB_PROJECT_ROOT)
        current_dir = COLAB_PROJECT_ROOT.resolve()
    else:
        raise RuntimeError(
            "Hãy mở notebook từ project root hoặc cd vào thư mục truck_blind_spot trước khi chạy."
        )

PROJECT_ROOT = current_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if IS_COLAB:
    run_command(["apt-get", "update", "-qq"])
    run_command(["apt-get", "install", "-y", "-qq", "libgl1", "libglib2.0-0"])
    if (PROJECT_ROOT / "requirements.txt").exists():
        run_command(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-r",
                str(PROJECT_ROOT / "requirements.txt"),
                "--quiet",
            ]
        )

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"Colab: {IS_COLAB}")


## 1. Kiểm tra GPU và cấu hình lệnh app.py

Cell này giữ cùng default với `app.py`. Nếu chưa bật GPU thì notebook dừng ngay.


In [ ]:
import torch

weights_path = "weights/best_roiv2.pt"
video_path = "assets/videos/demo4.mp4"
roi_config_path = "configs/roi.json"
roi_profile = "front_camera"
classes_config_path = "configs/classes.yaml"
conf_threshold = 0.25
iou_threshold = 0.45
prediction_horizon_s = 1.0
alert_threshold = 0.6

if not torch.cuda.is_available():
    raise RuntimeError(
        "Colab chưa bật GPU. Vào Runtime -> Change runtime type -> chọn GPU rồi chạy lại notebook."
    )
device = "0"


def require_file(relative_path: str) -> Path:
    """Kiểm tra file input của app.py tồn tại trong project."""
    resolved = PROJECT_ROOT / relative_path
    if not resolved.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {resolved}")
    return resolved


require_file("app.py")
require_file(weights_path)
require_file(video_path)
require_file(roi_config_path)
require_file(classes_config_path)

print(f"Device cho app.py: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Notebook sẽ chạy app.py trực tiếp, không lưu video output.")


## 2. Chạy app.py bằng GPU

Cell này gọi thẳng `app.main()`. Các hàm display của OpenCV được thay bằng no-op để pipeline chạy trọn vẹn trên GPU mà không cần cửa sổ X11:
- `cv2.imshow / namedWindow / resizeWindow / setWindowProperty / destroyAllWindows` → no-op
- `cv2.waitKey` → luôn trả về `-1`

Kết quả: mọi bước xử lý (detection, tracking, motion prediction) đều chạy; frame không hiển thị inline.


In [ ]:
import importlib
import sys
import cv2

# --- No-op patches để pipeline chạy không cần cửa sổ X11 ---
cv2.imshow              = lambda *a, **kw: None
cv2.namedWindow         = lambda *a, **kw: None
cv2.resizeWindow        = lambda *a, **kw: None
cv2.setWindowProperty   = lambda *a, **kw: None
cv2.waitKey             = lambda *a, **kw: -1
cv2.destroyAllWindows   = lambda: None

# --- Set sys.argv y hệt gọi từ terminal ---
sys.argv = [
    "app.py",
    "--source",             video_path,
    "--weights",            weights_path,
    "--roi",                roi_config_path,
    "--roi-profile",        roi_profile,
    "--classes-config",     classes_config_path,
    "--device",             device,
    "--conf-thres",         str(conf_threshold),
    "--iou-thres",          str(iou_threshold),
    "--prediction-horizon", str(prediction_horizon_s),
    "--alert-threshold",    str(alert_threshold),
]

# --- Import và chạy app.main() trực tiếp ---
import app as _app_module
importlib.reload(_app_module)  # reload để tránh state cũ khi chạy lại cell
_app_module.main()


## Lệnh tương đương khi chạy local

```bash
python app.py \
  --source assets/videos/demo4.mp4 \
  --weights weights/best_roiv2.pt \
  --roi configs/roi.json \
  --roi-profile front_camera \
  --classes-config configs/classes.yaml \
  --device 0
```

Trên Colab: notebook gọi `app.main()` trực tiếp sau khi thay `cv2.imshow` bằng no-op — cùng pipeline logic như chạy local, không cần màn hình.
